In [1]:
# ===================== Jupyter 超参数控制（直接在这里改）=====================
RUN_FULL_TEST = True   # False=极小样本试运行(5题/任务)  True=完整正式评测
# ==========================================================================

import os
import sys
import logging
import time
import subprocess
import json
import ssl
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context
from datetime import datetime
from tqdm import tqdm
import warnings
import urllib3
import shutil
import pathlib
import csv

# 屏蔽sklearn单一标签混淆矩阵警告
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings(
    "ignore",
    message="A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels."
)

# 关闭HF SSL证书校验，解决AutoDL自签名证书报错
os.environ["HF_HUB_SSL_VERIFY"] = "0"
os.environ["CURL_CA_BUNDLE"] = ""
# 全局urllib3关闭SSL警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ==========================================
# 1. 环境配置（修复冲突版本）
# ==========================================
# --- HuggingFace 镜像优先配置 ---
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# --- AutoDL 内网代理（仅联网下载阶段生效）
os.environ['http_proxy'] = 'http://10.37.1.23:12798'
os.environ['https_proxy'] = 'http://10.37.1.23:12798'
os.environ['HTTP_PROXY'] = 'http://10.37.1.23:12798'
os.environ['HTTPS_PROXY'] = 'http://10.37.1.23:12798'

# --- SSL 警告屏蔽
warnings.filterwarnings("ignore", message="Unverified HTTPS request")
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['HF_DATASETS_DISABLE_SSL_VERIFY'] = '1'

# --- HumanEval 允许执行模型生成的代码（pass@1 评估必需） ---
os.environ['HF_ALLOW_CODE_EVAL'] = '1'

# --- 关键修复：新版 huggingface_hub 1.22 改用 httpx，上面的 SSL 环境变量已失效 ---
# verify=False 解决 AutoDL 内网代理自签名证书；follow_redirects=True 解决 HuggingFaceH4/MATH-500
# 大小写规范 307 重定向导致的 JSONDecodeError
import httpx
from huggingface_hub import set_client_factory
set_client_factory(lambda: httpx.Client(verify=False, follow_redirects=True, timeout=httpx.Timeout(60.0)))

# --- 加速：hf-mirror 的 datasets-server 域名 SSL 握手失败会触发 5 次重试(共~23s/文件)
# AutoDL 代理对 datasets-server.hf-mirror.com 的 SSL 连接失败(503)，
# datasets 库会回退到 hf-mirror.com 直接下载 parquet 文件（正常工作）。
# 把 max_retries 从 5 降到 1，让失败的 HEAD 请求快速放弃走回退路径，整体提速 ~54%。
import huggingface_hub.utils._http as _hub_http
import huggingface_hub.file_download as _fd_mod
_orig_http_backoff = _hub_http.http_backoff
def _fast_http_backoff(method, url, **kwargs):
    kwargs['max_retries'] = 1
    return _orig_http_backoff(method=method, url=url, **kwargs)
_hub_http.http_backoff = _fast_http_backoff
if hasattr(_fd_mod, 'http_backoff'):
    _fd_mod.http_backoff = _fast_http_backoff

# --- vLLM 优化配置，关闭ModelScope强制数据源（解决404关键）
# os.environ["VLLM_USE_MODELSCOPE"] = "True"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# ==========================================
# 2. 安装缺失依赖
# ==========================================
def install_missing_dependencies():
    """安装缺失的 Python 依赖"""
    print("🔧 检查并安装缺失依赖...")
    
    try:
        import langdetect
        print("  ✅ langdetect 已安装")
    except ImportError:
        print("  📦 正在安装 langdetect...")
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', 'langdetect', '--quiet'],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            print("  ✅ langdetect 安装成功")
        except Exception as e:
            print(f"  ❌ langdetect 安装失败: {e}")
            print("  请手动运行: pip install langdetect")

# 模型与数据集路径
MODEL_NAME = "unsloth/Qwen3.5-4B"
LOCAL_MODEL_DIR = "./models/unsloth/Qwen3.5-4B"
DATA_ROOT = "./test-data"
LOG_FILE = f"benchmark_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
CSV_OUTPUT = f"bench_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# 评测任务列表（对齐 ModelScope 模型卡 benchmark，任务名用 lm-eval 标准名）
# 模型卡展示名 → lm-eval 任务名
#   MMLU-Redux → mmlu_redux_generative
#   MATH-500 → minerva_math500（4-shot CoT，不是 hendrycks_math500 的 0-shot）
#   IFEval → ifeval
#   C-Eval → ceval-valid
#   ARC-Easy/ARC-Challenge → arc_easy / arc_challenge
#   WinoGrande → winogrande
#   HellaSwag → hellaswag
#   Global PIQA → global_piqa_completions（136 种语言聚合任务）
#   BoolQ → boolq（super_glue 阅读理解二分类）
#   TruthfulQA-MC2 → truthfulqa_mc2（多选，测幻觉/真实性）
#   HumanEval → humaneval（Python 代码生成 pass@1）
#   BBH → bbh_cot_fewshot（Big-Bench-Hard 27 子任务，3-shot CoT）
TASKS = [
    "mmlu_redux_generative", "minerva_math500", "ifeval", "ceval-valid",
    "arc_easy", "arc_challenge", "winogrande", "hellaswag", "global_piqa_completions",
    "boolq", "truthfulqa_mc2", "humaneval", "bbh_cot_fewshot"
]

# 每个任务的样本配额（试运行模式生效；RUN_FULL_TEST=True 时忽略，跑全部题目）
TASK_LIMITS = {
    "mmlu_redux_generative": 10,
    "minerva_math500": 60,
    "ifeval": 45,
    "ceval-valid": 35,
    "arc_easy": 15,
    "arc_challenge": 15,
    "winogrande": 5,
    "hellaswag": 20,
    "global_piqa_completions": 15,
    "boolq": 10,
    "truthfulqa_mc2": 10,
    "humaneval": 20,
    "bbh_cot_fewshot": 10,
}

# ==========================================
# 3. 日志配置（修复重复打印日志）
# ==========================================
def setup_logger(log_file):
    logger = logging.getLogger("benchmark")
    logger.setLevel(logging.INFO)
    # 防止重复添加handler导致日志多行重复
    if not logger.handlers:
        fh = logging.FileHandler(log_file, encoding='utf-8')
        ch = logging.StreamHandler()
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        ch.setFormatter(formatter)
        logger.addHandler(fh)
        logger.addHandler(ch)
    return logger

logger = setup_logger(LOG_FILE)

# ==========================================
# 4. 数据集预加载（移除 trust_remote_code 参数，修复废弃警告）
# ==========================================
def preload_datasets(only_tasks=None):
    """预加载数据集到 datasets 库默认缓存目录
    only_tasks: 只加载指定的任务名列表（用于在线重试时跳过已缓存的数据集）
    
    通过 subprocess 调用 _download_datasets.py 下载，避开 notebook 里 set_client_factory
    等 monkey-patch 与 datasets 库冲突导致的 'maximum recursion depth exceeded' 问题。
    """
    import subprocess as _sp
    script = os.path.join(os.getcwd(), '_download_datasets.py')
    cmd = ["python", script] + (only_tasks if only_tasks else [])
    logger.info(f"🌐 通过 subprocess 预加载数据集{' (仅: '+str(only_tasks)+')' if only_tasks else ''}...")
    # 继承当前 env（含 HF_ENDPOINT、代理等），但关闭离线模式
    _env = os.environ.copy()
    _env.pop('HF_HUB_OFFLINE', None)
    _env.pop('HF_DATASETS_OFFLINE', None)
    try:
        _r = _sp.run(cmd, capture_output=True, text=True, env=_env, timeout=600)
        # 打印脚本输出（过滤进度条）
        for line in _r.stdout.splitlines():
            if line.strip() and 'examples/s' not in line and 'it/s' not in line and '%' not in line:
                logger.info(f"    {line}")
        if _r.returncode == 0:
            logger.info(f"📊 预加载完成")
            return True, []
        else:
            # 解析失败的任务名
            failed = []
            for line in _r.stderr.splitlines() + _r.stdout.splitlines():
                if '❌' in line and '失败' in line:
                    failed.append(line.split('❌')[1].split(' ')[0].strip())
            logger.error(f"❌ 预加载失败: {failed or '未知'}")
            return False, failed
    except _sp.TimeoutExpired:
        logger.error("❌ 预加载超时（600s）")
        return False, only_tasks or list(task_datasets_all.keys())

# ==========================================
# 5. 设置离线模式（预加载完成后执行，清除代理）
# ==========================================
def setup_offline_mode():
    """设置离线模式，强制 lm-eval 使用本地缓存，阻断在线查询"""
    logger.info("🔒 设置离线模式...")
    
    os.environ['HF_DATASETS_OFFLINE'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_HUB_ETAG_TIMEOUT'] = '0'
    os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '1'
    
    cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
    os.environ['HF_DATASETS_CACHE'] = cache_dir
    
    # 删除全部代理变量，离线不再走代理
    for var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY']:
        if var in os.environ:
            del os.environ[var]
    
    logger.info(f"  ✅ 离线模式已启用")
    logger.info(f"  ✅ 数据集缓存目录: {cache_dir}")

# ==========================================
# 6. 模型下载
# ==========================================
def download_model():
    """本地存在模型直接跳过下载"""
    if os.path.exists(LOCAL_MODEL_DIR) and os.listdir(LOCAL_MODEL_DIR):
        logger.info(f"✅ 本地模型已存在: {LOCAL_MODEL_DIR}")
        return LOCAL_MODEL_DIR
        
    logger.info(f"🌐 开始从 ModelScope 下载模型: {MODEL_NAME}")
    try:
        from modelscope import snapshot_download
        model_dir = snapshot_download(MODEL_NAME, cache_dir='./models')
        logger.info(f"✅ 模型下载完成: {model_dir}")
        return model_dir
    except Exception as e:
        logger.error(f"❌ 模型下载失败: {e}")
        return None

# ==========================================
# 7. 进度条监控类（Jupyter无清屏）
# ==========================================
class BenchmarkProgressTracker:
    def __init__(self, tasks_list):
        self.tasks_list = tasks_list
        self.task_progress = {}
        self.start_time = time.time()
        for task in tasks_list:
            self.task_progress[task] = {
                'status': 'pending',
                'start_time': None,
                'end_time': None,
                'samples_completed': 0,
                'total_samples': 0,
                'error': None
            }
    
    def update_task_status(self, task_name, status, error=None):
        if task_name in self.task_progress:
            self.task_progress[task_name]['status'] = status
            if status == 'running':
                self.task_progress[task_name]['start_time'] = time.time()
            elif status in ['completed', 'error']:
                self.task_progress[task_name]['end_time'] = time.time()
                if status == 'error':
                    self.task_progress[task_name]['error'] = error
    
    def update_sample_progress(self, task_name, completed, total):
        if task_name in self.task_progress:
            self.task_progress[task_name]['samples_completed'] = completed
            self.task_progress[task_name]['total_samples'] = total
    
    def print_progress(self):
        print("=" * 60)
        print("🚀 Qwen3.5-4B 基准测试进度监控")
        print("=" * 60)
        for task in self.tasks_list:
            info = self.task_progress[task]
            emoji = {'pending':'⏳','running':'🔄','completed':'✅','error':'❌'}.get(info['status'],'⏳')
            if info['status'] == 'running':
                if info['total_samples']>0:
                    pct = info['samples_completed'] / info['total_samples'] *100
                    bar = '█'*int(20*info['samples_completed']//info['total_samples']) + '-'*(20-int(20*info['samples_completed']//info['total_samples']))
                    print(f"{emoji} {task:15s} |{bar}| {pct:.1f}% ({info['samples_completed']}/{info['total_samples']})")
                else:
                    print(f"{emoji} {task:15s} | 正在初始化...")
            elif info['status'] == 'completed':
                dur = info['end_time'] - info['start_time'] if info['end_time'] and info['start_time'] else 0
                print(f"{emoji} {task:15s} | 完成 (耗时 {dur:.1f}s)")
            elif info['status'] == 'error':
                err = info['error'][:50] if info['error'] else '未知错误'
                print(f"{emoji} {task:15s} | 错误: {err}")
            else:
                print(f"{emoji} {task:15s} | 等待中")
        print("=" * 60)
        print(f"⏱️  总耗时: {time.time()-self.start_time:.1f}秒")
        print("=" * 60)

# ==========================================
# 8. 主执行逻辑（Jupyter直接运行）
# ==========================================
# 按任务配额：RUN_FULL_TEST=True 时跑全部，否则用 TASK_LIMITS 中每个任务的指定样本数
if RUN_FULL_TEST:
    logger.info("========== 运行模式: 正式测试 (全部题目) ==========")
else:
    logger.info(f"========== 运行模式: 按任务配额试运行 {TASK_LIMITS} ==========")

install_missing_dependencies()
logger.info(f"========== 开始评测模型: {MODEL_NAME} ==========")

# 预加载数据集：通过 subprocess 调用独立脚本下载，避开 notebook monkey-patch 冲突
# 已缓存的数据集会秒过（脚本内 load_dataset 直接命中缓存），未缓存的在线下载
all_dataset_loaded, failed_tasks = preload_datasets()
if failed_tasks:
    logger.warning(f"⚠️ 部分数据集下载失败，重试一次: {failed_tasks}")
    _, still_failed = preload_datasets(only_tasks=failed_tasks)
    all_dataset_loaded = len(still_failed) == 0
logger.info("⏳ 等待10秒释放镜像请求计数，避免429限流...")
time.sleep(10)

if not all_dataset_loaded:
    logger.warning("⚠️ 部分数据集预加载失败，离线模式下将无法运行对应任务")

# 开启离线
setup_offline_mode()

# 加载模型
model_path = download_model()
if not model_path:
    logger.error("模型获取失败，程序退出")
else:
    progress_tracker = BenchmarkProgressTracker(TASKS)

    try:
        # 修复 TritonTemplate / ExternKernelChoice 重复注册导致的 AssertionError
        # 根因：vllm 导入链触发 torch._inductor 模块级 TritonTemplate() / ExternKernelChoice() 创建，
        # Jupyter 重复运行或 torch._inductor 已被其他模块加载时，all_templates 已有同名条目 / extern_kernels 已有同名属性 → 断言失败
        # 方案1：TritonTemplate.all_templates 替换为允许覆盖的 dict 子类
        # 方案2：ExternKernelChoice.__init__ 调用前预清理 extern_kernels 上的同名属性，绕过断言
        try:
            from torch._inductor.select_algorithm import TritonTemplate, extern_kernels, ExternKernelChoice as _EKC
            class _AllowOverwriteDict(dict):
                def __contains__(self, key): return False
            if not isinstance(TritonTemplate.all_templates, _AllowOverwriteDict):
                TritonTemplate.all_templates = _AllowOverwriteDict(TritonTemplate.all_templates)
            _orig_ekc_init = _EKC.__init__
            def _patched_ekc_init(self, kernel, cpp_kernel=None, *, name=None, **kwargs):
                _name = name or getattr(kernel, '__name__', None)
                if _name and hasattr(extern_kernels, _name):
                    delattr(extern_kernels, _name)
                return _orig_ekc_init(self, kernel, cpp_kernel, name=name, **kwargs)
            _EKC.__init__ = _patched_ekc_init
        except Exception:
            pass
        from lm_eval import simple_evaluate
        from lm_eval.models.vllm_causallms import VLLM
    except ImportError as e:
        logger.error(f"导入评测库失败: {e}")
        logger.error("安装依赖：pip install vllm lm-eval tqdm datasets modelscope scikit-learn")
    else:
        logger.info("正在加载模型到 vLLM 引擎...")
        progress_tracker.print_progress()
        
        try:
            # 显存预检查：vLLM 需 gpu_memory_utilization*总显存，占用不足时提前报错
            import torch
            torch.cuda.empty_cache()
            _free_b, _total_b = torch.cuda.mem_get_info()
            _used_b = _total_b - _free_b
            _free_gib, _total_gib, _need_gib = _free_b/1024**3, _total_b/1024**3, 0.9*_total_b/1024**3
            logger.info(f"GPU 显存: 总 {_total_gib:.1f} GiB | 已用 {_used_b/1024**3:.1f} GiB | 可用 {_free_gib:.1f} GiB | vLLM 需 {_need_gib:.1f} GiB")
            if _free_gib < _need_gib:
                raise RuntimeError(f"显存不足: 可用 {_free_gib:.1f} GiB < 需求 {_need_gib:.1f} GiB。请关闭其他占用 GPU 的 notebook kernel 后重跑。当前 GPU 已用 {_used_b/1024**3:.1f}/{_total_gib:.1f} GiB")
            
            vllm_args = {
                "pretrained": model_path,
                "batch_size": 256,
                "trust_remote_code": True,
                "gpu_memory_utilization": 0.9,
                "max_model_len": 4096,
                "dtype": "auto"
            }
            model = VLLM(**vllm_args)
            logger.info("模型加载成功。")
        except Exception as e:
            logger.error(f"模型加载失败: {e}")
            progress_tracker.update_task_status("模型加载", 'error', str(e))
            progress_tracker.print_progress()
        else:
            logger.info(f"准备执行任务: {TASKS}")
            all_results = {}
            
            for task_name in TASKS:
                progress_tracker.update_task_status(task_name, 'running')
                progress_tracker.print_progress()
                # 按任务配额取 limit：正式测试跑全部(None)，试运行用 TASK_LIMITS 指定题数
                task_limit = None if RUN_FULL_TEST else TASK_LIMITS.get(task_name, 5)
                try:
                    results = simple_evaluate(
                        model=model,
                        tasks=[task_name],
                        limit=task_limit,
                        log_samples=False,
                        verbosity="WARNING",
                        confirm_run_unsafe_code=True  # humaneval 需要执行模型生成的代码
                    )
                    if results and "results" in results:
                        for res_task, metrics in results["results"].items():
                            all_results[res_task] = metrics
                    progress_tracker.update_task_status(task_name, 'completed')
                except Exception as e:
                    logger.error(f"任务 {task_name} 执行失败: {e}")
                    progress_tracker.update_task_status(task_name, 'error', str(e))
                progress_tracker.print_progress()
                time.sleep(0.5)
            
            # 控制台输出 + CSV导出
            print("\n" + "=" * 60)
            print("📊 评测结果汇总")
            print("=" * 60)
            
            if all_results:
                for task_name, metrics in all_results.items():
                    metric_strs = []
                    for k, v in metrics.items():
                        if k.startswith("alias") or k.startswith("n_"):
                            continue
                        if isinstance(v, float):
                            metric_strs.append(f"{k}: {v:.4f}")
                        elif isinstance(v, int) and not isinstance(v, bool):
                            metric_strs.append(f"{k}: {v}")
                    if metric_strs:
                        print(f"📌 {task_name:15s} | {' | '.join(metric_strs)}")
                
                # CSV写入：只输出基准类型和准确性得分（按TASKS聚合，子任务取平均）
                def _extract_acc(met):
                    """提取主准确率指标
                    各任务 metric 后缀差异：
                    - mmlu_redux: exact_match,default
                    - math500: exact_match,none + math_verify,none（取 math_verify）
                    - arc/hellaswag/global_piqa/cevalid: acc_norm,none（学术标准，长度归一化）优先；无则 acc,none
                    - boolq/truthfulqa/winogrande: acc,none（无 acc_norm）
                    - humaneval: pass@1,create_test
                    - bbh: exact_match,get-answer
                    - ifeval: prompt_level_strict_acc,none
                    """
                    # 精确匹配优先（metric 名含逗号后缀）；acc_norm,none 优先于 acc,none（arc/hellaswag/piqa 学术标准）
                    for k in ("pass@1,create_test", "math_verify,none", "acc_norm,none", "acc,none", "exact_match,default", "exact_match,none", "exact_match,get-answer", "prompt_level_strict_acc,none"):
                        if k in met and isinstance(met[k], (float, int)) and not isinstance(met[k], bool):
                            return met[k]
                    # 兜底：以 pass@1 开头的 metric（不同 lm-eval 版本后缀可能变化）
                    for k, v in met.items():
                        if k.startswith("pass@1") and isinstance(v, (float, int)) and not isinstance(v, bool):
                            return v
                    return None
                
                def _get_benchmark_type(res_task):
                    """将结果任务名映射到基准类型(TASKS中的名字)"""
                    if res_task.startswith("mmlu_redux"):
                        return "mmlu_redux_generative"
                    if res_task.startswith("ceval-valid"):
                        return "ceval-valid"
                    if res_task.startswith("global_piqa"):
                        return "global_piqa_completions"
                    if res_task.startswith("bbh_cot_fewshot"):
                        return "bbh_cot_fewshot"
                    return res_task  # minerva_math500, ifeval, arc_easy, arc_challenge, winogrande, hellaswag, boolq, truthfulqa_mc2, humaneval
                
                # 按基准类型聚合：聚合行(res_task==btype)直接用，否则子任务取平均
                btype_accs = {}
                for res_task, met in all_results.items():
                    btype = _get_benchmark_type(res_task)
                    acc = _extract_acc(met)
                    if acc is None:
                        continue
                    if btype not in btype_accs:
                        btype_accs[btype] = {"aggregate": None, "sub_accs": []}
                    if res_task == btype:
                        btype_accs[btype]["aggregate"] = acc
                    else:
                        btype_accs[btype]["sub_accs"].append(acc)
                
                with open(CSV_OUTPUT, "w", newline="", encoding="utf-8-sig") as f:
                    writer = csv.writer(f)
                    writer.writerow(["基准类型", "准确性得分(0-100)"])
                    for btype in TASKS:
                        if btype not in btype_accs:
                            continue
                        info = btype_accs[btype]
                        if info["aggregate"] is not None:
                            acc = info["aggregate"]
                        elif info["sub_accs"]:
                            acc = sum(info["sub_accs"]) / len(info["sub_accs"])
                        else:
                            continue
                        writer.writerow([btype, f"{acc * 100:.2f}"])
                print(f"\n✅ 结果已导出至CSV文件：{CSV_OUTPUT}")
            else:
                print("❌ 未能获取评测结果。请检查日志中的错误信息。")
            
            print("=" * 60)
            print(f"📁 详细日志已保存至: {LOG_FILE}")

2026-07-11 21:50:35,426 - INFO - ========== 运行模式: 正式测试 (全部题目) ==========
2026-07-11 21:50:35,432 - INFO - ========== 开始评测模型: unsloth/Qwen3.5-4B ==========
2026-07-11 21:50:35,433 - INFO - 🌐 通过 subprocess 预加载数据集...


🔧 检查并安装缺失依赖...
  ✅ langdetect 已安装


2026-07-11 21:53:19,182 - INFO -       - 下载 mmlu_redux_generative (fxmarty/mmlu-redux-2.0-ok, ALL)...
2026-07-11 21:53:19,183 - INFO -         📂 共 1 个 config，开始遍历...
2026-07-11 21:53:19,184 - INFO -         ✅ mmlu_redux_generative 已在缓存中
2026-07-11 21:53:19,185 - INFO -       - 下载 minerva_math500 (HuggingFaceH4/MATH-500, None)...
2026-07-11 21:53:19,186 - INFO -         ✅ minerva_math500 下载完成
2026-07-11 21:53:19,187 - INFO -       - 下载 ifeval (google/IFEval, None)...
2026-07-11 21:53:19,187 - INFO -         ✅ ifeval 下载完成
2026-07-11 21:53:19,188 - INFO -       - 下载 ceval-valid (ceval/ceval-exam, ALL)...
2026-07-11 21:53:19,189 - INFO -         📂 共 1 个 config，开始遍历...
2026-07-11 21:53:19,190 - INFO -         ✅ ceval-valid 已在缓存中
2026-07-11 21:53:19,191 - INFO -       - 下载 arc_easy (allenai/ai2_arc, ARC-Easy)...
2026-07-11 21:53:19,191 - INFO -         ✅ arc_easy 下载完成
2026-07-11 21:53:19,192 - INFO -       - 下载 arc_challenge (allenai/ai2_arc, ARC-Challenge)...
2026-07-11 21:53:19,193 - INFO 

🚀 Qwen3.5-4B 基准测试进度监控
⏳ mmlu_redux_generative | 等待中
⏳ minerva_math500 | 等待中
⏳ ifeval          | 等待中
⏳ ceval-valid     | 等待中
⏳ arc_easy        | 等待中
⏳ arc_challenge   | 等待中
⏳ winogrande      | 等待中
⏳ hellaswag       | 等待中
⏳ global_piqa_completions | 等待中
⏳ boolq           | 等待中
⏳ truthfulqa_mc2  | 等待中
⏳ humaneval       | 等待中
⏳ bbh_cot_fewshot | 等待中
⏱️  总耗时: 7.3秒
INFO 07-11 21:53:36 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'seed': 1234, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': './models/unsloth/Qwen3.5-4B'}
INFO 07-11 21:53:36 [model.py:598] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 07-11 21:53:36 [model.py:1725] Using max model len 4096
INFO 07-11 21:53:36 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-11 21:53:36 [vllm.py:1006] Asynchronous scheduling is enabled.
INFO 07-11 21:53:36 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPr

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

🚀 Qwen3.5-4B 基准测试进度监控
✅ mmlu_redux_generative | 完成 (耗时 187.6s)
✅ minerva_math500 | 完成 (耗时 60.7s)
✅ ifeval          | 完成 (耗时 120.3s)
✅ ceval-valid     | 完成 (耗时 53.5s)
✅ arc_easy        | 完成 (耗时 53.1s)
✅ arc_challenge   | 完成 (耗时 33.7s)
✅ winogrande      | 完成 (耗时 21.3s)
✅ hellaswag       | 完成 (耗时 279.8s)
✅ global_piqa_completions | 完成 (耗时 149.6s)
✅ boolq           | 完成 (耗时 81.9s)
✅ truthfulqa_mc2  | 完成 (耗时 86.6s)
⏳ humaneval       | 等待中
⏳ bbh_cot_fewshot | 等待中
⏱️  总耗时: 1189.0秒
🚀 Qwen3.5-4B 基准测试进度监控
✅ mmlu_redux_generative | 完成 (耗时 187.6s)
✅ minerva_math500 | 完成 (耗时 60.7s)
✅ ifeval          | 完成 (耗时 120.3s)
✅ ceval-valid     | 完成 (耗时 53.5s)
✅ arc_easy        | 完成 (耗时 53.1s)
✅ arc_challenge   | 完成 (耗时 33.7s)
✅ winogrande      | 完成 (耗时 21.3s)
✅ hellaswag       | 完成 (耗时 279.8s)
✅ global_piqa_completions | 完成 (耗时 149.6s)
✅ boolq           | 完成 (耗时 81.9s)
✅ truthfulqa_mc2  | 完成 (耗时 86.6s)
🔄 humaneval       | 正在初始化...
⏳ bbh_cot_fewshot | 等待中
⏱️  总耗时: 1189.5秒


Using the latest cached version of the dataset since openai/openai_humaneval couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'openai_humaneval' at /root/.cache/huggingface/datasets/openai___openai_humaneval/openai_humaneval/0.0.0/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544 (last modified on Sat Jul 11 12:13:08 2026).
Running generate_until requests: 100%|██████████| 164/164 [00:09<00:00, 17.99it/s]


🚀 Qwen3.5-4B 基准测试进度监控
✅ mmlu_redux_generative | 完成 (耗时 187.6s)
✅ minerva_math500 | 完成 (耗时 60.7s)
✅ ifeval          | 完成 (耗时 120.3s)
✅ ceval-valid     | 完成 (耗时 53.5s)
✅ arc_easy        | 完成 (耗时 53.1s)
✅ arc_challenge   | 完成 (耗时 33.7s)
✅ winogrande      | 完成 (耗时 21.3s)
✅ hellaswag       | 完成 (耗时 279.8s)
✅ global_piqa_completions | 完成 (耗时 149.6s)
✅ boolq           | 完成 (耗时 81.9s)
✅ truthfulqa_mc2  | 完成 (耗时 86.6s)
✅ humaneval       | 完成 (耗时 70.0s)
⏳ bbh_cot_fewshot | 等待中
⏱️  总耗时: 1259.5秒
🚀 Qwen3.5-4B 基准测试进度监控
✅ mmlu_redux_generative | 完成 (耗时 187.6s)
✅ minerva_math500 | 完成 (耗时 60.7s)
✅ ifeval          | 完成 (耗时 120.3s)
✅ ceval-valid     | 完成 (耗时 53.5s)
✅ arc_easy        | 完成 (耗时 53.1s)
✅ arc_challenge   | 完成 (耗时 33.7s)
✅ winogrande      | 完成 (耗时 21.3s)
✅ hellaswag       | 完成 (耗时 279.8s)
✅ global_piqa_completions | 完成 (耗时 149.6s)
✅ boolq           | 完成 (耗时 81.9s)
✅ truthfulqa_mc2  | 完成 (耗时 86.6s)
✅ humaneval       | 完成 (耗时 70.0s)
🔄 bbh_cot_fewshot | 正在初始化...
⏱️  总耗时: 1260.0秒


Using the latest cached version of the dataset since SaylorTwift/bbh couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'boolean_expressions' at /root/.cache/huggingface/datasets/SaylorTwift___bbh/boolean_expressions/0.0.0/b5306be6f827cfafbb545ff5a51f96916029b0fd (last modified on Sat Jul 11 12:30:18 2026).
Using the latest cached version of the dataset since SaylorTwift/bbh couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'causal_judgement' at /root/.cache/huggingface/datasets/SaylorTwift___bbh/causal_judgement/0.0.0/b5306be6f827cfafbb545ff5a51f96916029b0fd (last modified on Sat Jul 11 12:30:34 2026).
Using the latest cached version of the dataset since SaylorTwift/bbh couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'date_understanding' at /root/.cache/huggingface/datasets/SaylorTwift___bbh/

🚀 Qwen3.5-4B 基准测试进度监控
✅ mmlu_redux_generative | 完成 (耗时 187.6s)
✅ minerva_math500 | 完成 (耗时 60.7s)
✅ ifeval          | 完成 (耗时 120.3s)
✅ ceval-valid     | 完成 (耗时 53.5s)
✅ arc_easy        | 完成 (耗时 53.1s)
✅ arc_challenge   | 完成 (耗时 33.7s)
✅ winogrande      | 完成 (耗时 21.3s)
✅ hellaswag       | 完成 (耗时 279.8s)
✅ global_piqa_completions | 完成 (耗时 149.6s)
✅ boolq           | 完成 (耗时 81.9s)
✅ truthfulqa_mc2  | 完成 (耗时 86.6s)
✅ humaneval       | 完成 (耗时 70.0s)
✅ bbh_cot_fewshot | 完成 (耗时 613.6s)
⏱️  总耗时: 1873.6秒

📊 评测结果汇总
📌 mmlu_redux_abstract_algebra_generative | sample_len: 89 | exact_match,default: 0.3034 | exact_match_stderr,default: 0.0490
📌 mmlu_redux_anatomy_generative | sample_len: 99 | exact_match,default: 0.3535 | exact_match_stderr,default: 0.0483
📌 mmlu_redux_astronomy_generative | sample_len: 94 | exact_match,default: 0.2979 | exact_match_stderr,default: 0.0474
📌 mmlu_redux_college_biology_generative | sample_len: 98 | exact_match,default: 0.3265 | exact_match_stderr,default: 0.0476
📌 mmlu_